# Differentiable Optimization Showcases

Two optimization problems that leverage end-to-end differentiability through the gyrokinetic solver:

1. **Equilibrium design** — optimize magnetic shear $\hat{s}$ to achieve a target heat flux level, relevant to tokamak configuration optimization.
2. **Optimal initial condition** — find the perturbation that reaches nonlinear saturation fastest, testing AD through the full RK4 integration.

Both use ITG-like parameters from iteration\_13 ($q=4.57$, $R/L_T=10.2$) on a reduced grid.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import logging

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_command_buffer="
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("jax._src.xla_bridge").setLevel(logging.ERROR)

import sys

sys.path.append("..")

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from gyaradax import compute_geometry, GKParams, gk_init, gksolve
from gyaradax.solver import linear_precompute, gkstep_single, default_state
from gyaradax.integrals import get_integrals
from gyaradax.plot_utils import JAX_COLORS

jax.config.update("jax_enable_x64", True)

## Reduced grid convergence check

We use iteration\_13 physical parameters on a reduced grid with enough modes for nonlinear saturation. The reduced grid won't match the full-resolution flux level exactly, but should show a stable saturated state.

In [ ]:
# iteration_13 physical parameters
Q, SHAT, EPS = 4.5684, 3.0751, 0.19
RLT, RLN = 10.1736, 2.6088

# larger reduced grid for nonlinear saturation
NS, NKX, NKY, NVPAR, NMU = 16, 33, 16, 24, 8
DT = 0.01
NAVERAGE = 120
N_STEPS = 4000  # long enough for saturation
KXMAX = 2.0 * NKX * SHAT

geometry = compute_geometry(
    q=Q,
    shat=SHAT,
    eps=EPS,
    ns=NS,
    nkx=NKX,
    nky=NKY,
    nvpar=NVPAR,
    nmu=NMU,
    vpar_max=3.0,
    nperiod=1,
    krhomax=1.4,
    kxmax=KXMAX,
)
ky = np.asarray(geometry["krho"])
print(f"grid: ({NVPAR}, {NMU}, {NS}, {NKX}, {NKY})")
print(f"ky range: [{ky[0]:.3f}, {ky[-1]:.3f}]")

# Grid-derived constants (concrete, not traced through AD)
_DVP = float(geometry["dvp"])
_SGR_DIST = float(geometry["sgr_dist"])
_KTHNORM = float(geometry["kthnorm"])
_RREF = float(geometry["Rref"])
_D2X = float(geometry["d2X"])
_SIGNB = float(geometry["signB"])
_KYMAX = float(geometry["kymax"])
_KXMAX = float(geometry["kxmax"])


def make_params(rlt=RLT, rln=RLN, disp=0.1, non_linear=True, mixed_precision=True):
    """All params are concrete floats — geometry dependence flows through compute_geometry only."""
    return GKParams(
        dt=DT,
        naverage=NAVERAGE,
        rlt=rlt,
        rln=rln,
        disp_par=disp,
        disp_vp=disp,
        disp_x=disp,
        disp_y=disp,
        non_linear=non_linear,
        finit="cosine2",
        mixed_precision=mixed_precision,
        shat=SHAT,
        q=Q,
        eps=EPS,
        kthnorm=_KTHNORM,
        Rref=_RREF,
        d2X=_D2X,
        signB=_SIGNB,
        dvp=_DVP,
        sgr_dist=_SGR_DIST,
        kxmax=_KXMAX,
        kymax=_KYMAX,
    )

## 1. Geometry Optimization: Minimize Heat Flux by Tuning $(q, \hat{s}, \varepsilon)$

Given a fixed ITG drive ($R/L_T = 10.2$, $R/L_n = 2.6$), we optimize the magnetic geometry to minimize turbulent heat flux. The three geometry parameters are:
- $q$ — safety factor (toroidal winding number)
- $\hat{s} = (r/q)\,dq/dr$ — magnetic shear (radial variation of $q$)
- $\varepsilon = r/R_0$ — inverse aspect ratio (plasma shape)

Since `compute_geometry` is now fully JAX-differentiable, gradients flow end-to-end from the loss through the solver and geometry arrays back to $(q, \hat{s}, \varepsilon)$.

**Truncated backpropagation.** Differentiating through a chaotic turbulent trajectory causes the adjoint state to diverge exponentially (the Lyapunov exponent amplifies the backward pass). We address this by running the simulation to saturation *without* gradients, then differentiating only through a short window at the end. This gives the local sensitivity of the saturated flux to geometry parameters.

In [ ]:
# Forward model: (q, shat, eps) -> saturated heat flux
# Two-phase: warmup outside AD, then differentiable grad window.

WARMUP_STEPS = 120 * 20
GRAD_STEPS = 120 * 10
GRAD_BLOCK = 40


def warmup_to_saturation(q_val, shat_val, eps_val):
    """Run to saturation WITHOUT gradients. Returns (df, state).
    JIT compiles via lax.scan internally; recompiles only when geometry changes."""
    geom = compute_geometry(
        q=q_val,
        shat=shat_val,
        eps=eps_val,
        ns=NS,
        nkx=NKX,
        nky=NKY,
        nvpar=NVPAR,
        nmu=NMU,
        vpar_max=3.0,
        nperiod=1,
        krhomax=1.4,
        kxmax=_KXMAX,
    )
    params = make_params(non_linear=True, mixed_precision=True)
    df0, _, state0 = gk_init(geom, params)
    pre = linear_precompute(geom, params)

    @jax.jit
    def _scan(df, state):
        def step(carry, _):
            d, s = carry
            d, _, s = gkstep_single(d, geom, params, s, pre)
            return (d, s), None

        return jax.lax.scan(step, (df, state), None, length=WARMUP_STEPS)

    (df, state), _ = _scan(df0, state0)
    return df, state


def grad_window(df_warm, state_warm, q_val, shat_val, eps_val):
    """Short differentiable window. Geometry is recomputed with traced params."""
    geom = compute_geometry(
        q=q_val,
        shat=shat_val,
        eps=eps_val,
        ns=NS,
        nkx=NKX,
        nky=NKY,
        nvpar=NVPAR,
        nmu=NMU,
        vpar_max=3.0,
        nperiod=1,
        krhomax=1.4,
        kxmax=_KXMAX,
    )
    params = make_params(non_linear=True, mixed_precision=False)
    pre = linear_precompute(geom, params)

    @jax.jit
    @jax.checkpoint
    def block(carry, _):
        def step(c, _):
            d, s = c
            d, _, s = gkstep_single(d, geom, params, s, pre)
            return (d, s), None

        return jax.lax.scan(step, carry, None, length=GRAD_BLOCK)

    (df, _), _ = jax.lax.scan(block, (df_warm, state_warm), None, length=GRAD_STEPS // GRAD_BLOCK)
    _, fluxes = get_integrals(df, geom, params=params, pre=pre)
    return fluxes[1]


# Phase 1: warmup (concrete, no grad)
print(f"warmup: {WARMUP_STEPS} steps...")
t0 = time.time()
q0, shat0, eps0 = jnp.array(Q), jnp.array(SHAT), jnp.array(EPS)
df_sat, state_sat = warmup_to_saturation(Q, SHAT, EPS)
print(f"done ({time.time()-t0:.1f}s)")

# Phase 2: forward through grad window
print(f"\ncompiling grad window ({GRAD_STEPS} steps, block={GRAD_BLOCK})...")
t0 = time.time()
ef_baseline = grad_window(df_sat, state_sat, q0, shat0, eps0)
print(f"eflux = {float(ef_baseline):.4f} ({time.time()-t0:.1f}s)")

In [ ]:
# Convenience wrapper for the full forward pass (warmup + grad window)
def eflux_of_geometry(q_val, shat_val, eps_val):
    """Full forward: warmup to saturation, then grad window. Not differentiable as a whole."""
    df, state = warmup_to_saturation(float(q_val), float(shat_val), float(eps_val))
    return grad_window(df, state, q_val, shat_val, eps_val)


# Compute gradient w.r.t. geometry params through the grad window only
grad_fn = jax.grad(lambda q, s, e: grad_window(df_sat, state_sat, q, s, e), argnums=(0, 1, 2))

print("compiling grad...")
t0 = time.time()
dQ_dq, dQ_dshat, dQ_deps = grad_fn(q0, shat0, eps0)
print(f"done ({time.time()-t0:.1f}s)")
print(f"  dQ/dq    = {float(dQ_dq):.4e}")
print(f"  dQ/dshat = {float(dQ_dshat):.4e}")
print(f"  dQ/deps  = {float(dQ_deps):.4e}")

Q0 = float(ef_baseline)
print(f"\nnormalized sensitivities (p * dQ/dp / Q):")
print(f"  q:    {float(q0 * dQ_dq / Q0):.3f}")
print(f"  shat: {float(shat0 * dQ_dshat / Q0):.3f}")
print(f"  eps:  {float(eps0 * dQ_deps / Q0):.3f}")

In [ ]:
N_ITERS = 20
LR = 0.005

params_vec = jnp.array([Q, SHAT, EPS])
optimizer = optax.adam(learning_rate=LR)
opt_state = optimizer.init(params_vec)

BOUNDS_LO = jnp.array([1.0, 0.3, 0.05])
BOUNDS_HI = jnp.array([8.0, 5.0, 0.35])

history_geom = {"params": [np.array(params_vec)], "loss": []}

pbar = tqdm(range(N_ITERS), desc="optimizing geometry")
for i in pbar:
    q_c, s_c, e_c = float(params_vec[0]), float(params_vec[1]), float(params_vec[2])
    df_w, st_w = warmup_to_saturation(q_c, s_c, e_c)

    loss_and_grad = jax.value_and_grad(lambda p: grad_window(df_w, st_w, p[0], p[1], p[2]))
    l, g = loss_and_grad(params_vec)
    l = float(l)

    updates, opt_state = optimizer.update(g, opt_state)
    params_vec = optax.apply_updates(params_vec, updates)
    params_vec = jnp.clip(params_vec, BOUNDS_LO, BOUNDS_HI)

    history_geom["params"].append(np.asarray(params_vec))
    history_geom["loss"].append(l)
    pbar.set_postfix(loss=f"{l:.2f}", q=f"{q_c:.2f}", s=f"{s_c:.2f}", e=f"{e_c:.3f}")

final = params_vec
print(f"\noptimized geometry:")
print(f"  q:    {Q:.3f} -> {float(final[0]):.3f}")
print(f"  shat: {SHAT:.3f} -> {float(final[1]):.3f}")
print(f"  eps:  {EPS:.3f} -> {float(final[2]):.3f}")
print(f"  flux: {float(ef_baseline):.2f} -> {history_geom['loss'][-1]:.2f}")

In [ ]:
param_history = np.array(history_geom["params"])
loss_history = np.array(history_geom["loss"])
q_i, s_i, e_i = param_history[0]
q_f, s_f, e_f = param_history[-1]

print("running before/after simulations...")


def run_flux_trace(q_val, shat_val, eps_val, n_blocks=100):
    geom = compute_geometry(
        q=q_val,
        shat=shat_val,
        eps=eps_val,
        ns=NS,
        nkx=NKX,
        nky=NKY,
        nvpar=NVPAR,
        nmu=NMU,
        vpar_max=3.0,
        nperiod=1,
        krhomax=1.4,
        kxmax=_KXMAX,
    )
    params = make_params(non_linear=True, mixed_precision=True)
    df, _, state = gk_init(geom, params)
    pre = linear_precompute(geom, params)
    times, efluxes = [0.0], [0.0]
    last_phi = None
    for b in range(n_blocks):
        df, (phi, fluxes), state = gksolve(df, geom, params, state, n_steps=NAVERAGE, pre=pre)
        last_phi = phi
        times.append(float(state.time))
        efluxes.append(float(fluxes[1]))
        if (b + 1) % 10 == 0:
            print(f"  [{b+1}/{n_blocks}] t={times[-1]:.1f} Q={efluxes[-1]:.2f}")
    ky_spec = np.sum(np.abs(np.asarray(last_phi)) ** 2, axis=(0, 1))
    return np.array(times), np.array(efluxes), ky_spec


print(f"initial geometry (q={q_i:.2f}, s={s_i:.2f}, e={e_i:.3f}):")
t_init, ef_init, spec_init = run_flux_trace(q_i, s_i, e_i)
print(f"optimized geometry (q={q_f:.2f}, s={s_f:.2f}, e={e_f:.3f}):")
t_opt, ef_opt, spec_opt = run_flux_trace(q_f, s_f, e_f)
print("done")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.0))

# saturated flux from actual simulations (last third)
avg_n = max(1, len(ef_init) // 3)
Q_init_sat = np.mean(ef_init[-avg_n:])
Q_opt_sat = np.mean(ef_opt[-avg_n:])
flux_change = (Q_opt_sat - Q_init_sat) / Q_init_sat * 100

# (a) flux convergence (grad window loss — local sensitivity)
ax = axes[0, 0]
ax.plot(loss_history, "o-", color=JAX_COLORS["cyan"], ms=4, lw=1.2)
ax.set_xlabel("iteration", fontsize=13)
ax.set_ylabel("grad window flux", fontsize=13)
ax.set_title("(a) optimizer loss", fontsize=15)
ax.grid(True)

# (b) parameter trajectories (normalized)
ax = axes[0, 1]
labels_p = [r"$q$", r"$\hat{s}$", r"$\varepsilon$"]
colors_p = [JAX_COLORS["blue"], JAX_COLORS["red"], JAX_COLORS["green"]]
for j in range(3):
    trajectory = param_history[:, j] / param_history[0, j]
    ax.plot(trajectory, "o-", color=colors_p[j], ms=3, lw=1, label=labels_p[j])
ax.axhline(1.0, color="k", ls="--", lw=0.6, alpha=0.4)
ax.set_xlabel("iteration", fontsize=13)
ax.set_ylabel("relative to initial", fontsize=13)
ax.set_title("(b) geometry parameters", fontsize=15)
ax.legend(frameon=False, fontsize=12)
ax.grid(True)

# (c) before/after flux time traces with saturated averages
ax = axes[1, 0]
ax.plot(t_init, ef_init, "-", color=JAX_COLORS["cyan"], lw=1, alpha=0.5, label="initial")
ax.plot(t_opt, ef_opt, "-", color=JAX_COLORS["purple"], lw=1.2, label="optimized")
ax.axhline(Q_init_sat, color=JAX_COLORS["cyan"], ls="--", lw=0.8, alpha=0.6)
ax.axhline(Q_opt_sat, color=JAX_COLORS["purple"], ls="--", lw=0.8)
ax.set_xlabel(r"time $[R/v_\mathrm{th}]$", fontsize=13)
ax.set_ylabel("heat flux $Q$", fontsize=13)
ax.set_title("(c) flux time traces", fontsize=15)
ax.legend(frameon=False, fontsize=12, loc="lower right")
ax.grid(True)

# (d) before/after ky spectra
ax = axes[1, 1]
ax.semilogy(ky, spec_init, "o-", color=JAX_COLORS["cyan"], ms=3, lw=1, alpha=0.6, label="initial")
ax.semilogy(ky, spec_opt, "o-", color=JAX_COLORS["purple"], ms=3, lw=1.2, label="optimized")
ax.set_xlabel(r"$k_y \rho_\mathrm{ref}$", fontsize=13)
ax.set_ylabel(r"$k_y$ spectrum", fontsize=13)
ax.set_title(r"(d) turbulence spectrum", fontsize=15)
# ax.legend(frameon=False, fontsize=12)
ax.grid(True, which="both")

sign = "+" if flux_change > 0 else ""
fig.suptitle(
    rf"Geometry: $q$ {q_i:.1f}$\to${q_f:.1f}, $\hat{{s}}$ {s_i:.1f}$\to${s_f:.1f},"
    rf" $\varepsilon$ {e_i:.2f}$\to${e_f:.2f}"
    rf" — saturated $Q$: {Q_init_sat:.1f}$\to${Q_opt_sat:.1f} ({sign}{flux_change:.0f}%)",
    fontsize=12,
    y=1.02,
)

fig.tight_layout()
fig.savefig("figs/opt_geometry.pdf")
plt.show()

## 2. Optimal Initial Condition for Fastest Growth

We optimize the initial perturbation amplitude profile $a(k_y)$ to maximize the mode energy at time $T$, finding the perturbation that excites the instability most efficiently. This tests AD through the full RK4 `lax.scan` integration in the linear regime.

The loss is the negative total spectral energy at the end of the simulation:
$$\mathcal{L}(\mathbf{a}) = -\sum_{k_y} |\phi(T; \mathbf{a})|^2$$

where $\mathbf{a} \in \mathbb{R}^{N_{k_y}}$ modulates the per-$k_y$ initial amplitude. The optimizer discovers which $k_y$ modes grow fastest under the ITG instability.

In [ ]:
N_STEPS_IC = 200  # shorter horizon for IC optimization

# fixed params (linear, baseline shat)
params_ic = make_params(non_linear=False)
pre_ic = linear_precompute(geometry, params_ic)

# base init: cosine2 profile
df_base, _, state_base = gk_init(geometry, params_ic)
print(f"df_base shape: {df_base.shape}")  # (nvpar, nmu, ns, nkx, nky)


def energy_of_ky_amps(log_amps):
    """Run solver with per-ky amplitude modulation, return total phi energy.

    log_amps: (NKY,) array of log-scale amplitude factors.
    """
    amps = jnp.exp(log_amps)  # ensure positivity
    # broadcast (NKY,) -> (1, 1, 1, 1, NKY) to scale each ky slice
    df_mod = df_base * amps[None, None, None, None, :]

    _, (phi, _), _ = gksolve(
        df_mod, geometry, params_ic, state_base, n_steps=N_STEPS_IC, pre=pre_ic
    )
    return jnp.sum(jnp.abs(phi) ** 2)


# maximize energy = minimize negative energy
def loss_ic(log_amps):
    return -energy_of_ky_amps(log_amps)


grad_ic = jax.jit(jax.grad(loss_ic))

# compile
print("compiling IC forward + backward...")
t0 = time.time()
init_log_amps = jnp.zeros(NKY)
e0 = energy_of_ky_amps(init_log_amps)
_ = grad_ic(init_log_amps)
print(f"done ({time.time()-t0:.1f}s)")
print(f"baseline energy: {float(e0):.4e}")

# check gradient is nonzero
g0 = np.asarray(grad_ic(init_log_amps))
print(f"grad at baseline: min={g0.min():.2e} max={g0.max():.2e} norm={np.linalg.norm(g0):.2e}")

In [ ]:
N_ITERS_IC = 80
LR_IC = 0.1

log_amps = jnp.zeros(NKY)
optimizer_ic = optax.adam(learning_rate=LR_IC)
opt_state_ic = optimizer_ic.init(log_amps)

history_ic = {"loss": [], "energy": [], "amps": [np.ones(NKY)]}

pbar = tqdm(range(N_ITERS_IC), desc="optimizing IC")
for i in pbar:
    l = float(loss_ic(log_amps))
    g = grad_ic(log_amps)
    updates, opt_state_ic = optimizer_ic.update(g, opt_state_ic)
    log_amps = optax.apply_updates(log_amps, updates)
    log_amps = jnp.clip(log_amps, -5.0, 5.0)

    energy = float(energy_of_ky_amps(log_amps))
    history_ic["loss"].append(l)
    history_ic["energy"].append(energy)
    history_ic["amps"].append(np.asarray(jnp.exp(log_amps)))
    pbar.set_postfix(loss=f"{l:.2e}", energy=f"{energy:.2e}")

print(
    f"\nfinal energy: {history_ic['energy'][-1]:.4e} (baseline: {float(energy_of_ky_amps(jnp.zeros(NKY))):.4e})"
)
print(f"amplification: {history_ic['energy'][-1] / float(energy_of_ky_amps(jnp.zeros(NKY))):.1f}x")

In [ ]:
# Run before/after IC simulations for comparison
print("running uniform vs optimized IC...")


def run_energy_trace(log_amps_val, n_steps=N_STEPS_IC):
    amps = jnp.exp(log_amps_val)
    df_mod = df_base * amps[None, None, None, None, :]
    state = state_base
    df = df_mod
    energies = []
    step_size = 10
    for b in range(n_steps // step_size):
        df, (phi, _), state = gksolve(df, geometry, params_ic, state, n_steps=step_size, pre=pre_ic)
        energies.append(float(jnp.sum(jnp.abs(phi) ** 2)))
    return np.array(energies)


def get_final_ky_spec(log_amps_val):
    amps = jnp.exp(log_amps_val)
    df_mod = df_base * amps[None, None, None, None, :]
    _, (phi, _), _ = gksolve(
        df_mod, geometry, params_ic, state_base, n_steps=N_STEPS_IC, pre=pre_ic
    )
    return np.sum(np.abs(np.asarray(phi)) ** 2, axis=(0, 1))


energy_uniform = run_energy_trace(jnp.zeros(NKY))
energy_optimal = run_energy_trace(log_amps)
spec_uniform = get_final_ky_spec(jnp.zeros(NKY))
spec_optimal = get_final_ky_spec(log_amps)
t_steps = np.arange(1, len(energy_uniform) + 1) * 10 * params_ic.dt
print("done")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.0))

# (a) energy convergence over optimizer iterations
ax = axes[0, 0]
ax.semilogy(history_ic["energy"], "o-", color=JAX_COLORS["cyan"], ms=3, lw=1)
ax.axhline(
    float(energy_of_ky_amps(jnp.zeros(NKY))), color="k", ls="--", lw=0.8, label="uniform init"
)
ax.set_xlabel("iteration", fontsize=13)
ax.set_ylabel(r"$\sum |\phi|^2$ at $T$", fontsize=13)
ax.set_title("(a) energy maximization", fontsize=15)
ax.legend(frameon=False, fontsize=12)
ax.grid(True, which="both")

# (b) optimal amplitude profile
ax = axes[0, 1]
final_amps = history_ic["amps"][-1]
ax.bar(range(NKY), final_amps, color=JAX_COLORS["purple"], alpha=0.8)
ax.axhline(1.0, color="k", ls="--", lw=0.6, alpha=0.4, label="uniform")
ax.set_xticks(range(0, NKY, 2))
ax.set_xticklabels([f"{ky[k]:.2f}" for k in range(0, NKY, 2)], fontsize=9)
ax.set_xlabel(r"$k_y \rho_\mathrm{ref}$", fontsize=13)
ax.set_ylabel("amplitude factor", fontsize=13)
ax.set_title("(b) optimal $a(k_y)$", fontsize=15)
ax.legend(frameon=False, fontsize=12)
ax.grid(True, axis="y")

# (c) energy evolution: uniform vs optimized IC
ax = axes[1, 0]
ax.semilogy(
    t_steps, energy_uniform, "-", color=JAX_COLORS["cyan"], lw=1, alpha=0.6, label="uniform IC"
)
ax.semilogy(t_steps, energy_optimal, "-", color=JAX_COLORS["purple"], lw=1.2, label="optimized IC")
ax.set_xlabel(r"time $[R/v_\mathrm{th}]$", fontsize=13)
ax.set_ylabel(r"$\sum |\phi|^2$", fontsize=13)
ax.set_title("(c) energy growth", fontsize=15)
ax.legend(frameon=False, fontsize=12)
ax.grid(True, which="both")

# (d) ky spectrum at T: uniform vs optimized
ax = axes[1, 1]
ax.semilogy(
    ky, spec_uniform, "o-", color=JAX_COLORS["cyan"], ms=3, lw=1, alpha=0.6, label="uniform IC"
)
ax.semilogy(ky, spec_optimal, "o-", color=JAX_COLORS["purple"], ms=3, lw=1.2, label="optimized IC")
ax.set_xlabel(r"$k_y \rho_\mathrm{ref}$", fontsize=13)
ax.set_ylabel(r"$k_y$ spectrum at $T$", fontsize=13)
ax.set_title("(d) spectral redistribution", fontsize=15)
ax.legend(frameon=False, fontsize=12)
ax.grid(True, which="both")

amplification = history_ic["energy"][-1] / float(energy_of_ky_amps(jnp.zeros(NKY)))
fig.suptitle(
    f"{amplification:.0f}$\\times$ energy amplification via optimal seeding", fontsize=13, y=1.02
)

fig.tight_layout()
fig.savefig("figs/opt_ic.pdf")
plt.show()